In [41]:
import time
import json
import random

import cst_python as cst
from cst_python.memory_storage import MemoryStorageCodelet

# Inicia a mente

In [42]:
mind = cst.Mind()

surrounding_actions_mo = mind.create_memory_object("SurroundingActions")
skill_manifest_mo = mind.create_memory_object("SkillManifest")
action_command_mo = mind.create_memory_object("ActionCommand")
action_status_mo = mind.create_memory_object("ActionStatus")

mscodelet = MemoryStorageCodelet(mind, host="127.0.0.1")
mscodelet.time_step = 50
mind.insert_codelet(mscodelet)

mind.start()

In [43]:
while surrounding_actions_mo.get_info() == "" and skill_manifest_mo.get_info() == "":
    time.sleep(1)

# Prepara os comandos

In [44]:
from dataclasses import dataclass, field, asdict
from typing import Any


@dataclass
class CommandEntry:
    Skill : str
    Parameters:dict[str, Any]=field(default_factory=dict)

@dataclass
class CommandPayload:
    Id:int
    Commands:list[CommandEntry]
    

In [45]:
velocity = 3.5

def create_commands(surrounding_actions:dict) -> dict[str, list[CommandEntry]]:
    result = {}

    for action in surrounding_actions:
        commands = []
        position = action["originPosition"]
        name = action["name"]

        commands.append(CommandEntry("walk_to", 
                                    {"destination":position, "velocity":velocity}))

        if name == "Trabalhar":
            commands.append(CommandEntry("work"))

        elif name == "Beber café":
            commands.append(CommandEntry("drink_coffee"))

        elif name == "Usar":
            commands.append(CommandEntry("use_bathroom"))

        result[name] = commands

    return result

In [46]:
commands = create_commands(surrounding_actions_mo.get_info())

# Loop para selecionar ações

In [47]:
payload = CommandPayload(1, commands["Beber café"])

action_command_mo.set_info(asdict(payload))

-1

In [48]:
last_id = 1
actions = list(commands.keys())

last_payload_size = 2

while True:
    # Aguarda até que action_status_mo tenha conteúdo
    info = action_status_mo.get_info()
    while info == "" or info is None:
        time.sleep(1)
        info = action_status_mo.get_info()

    status = json.loads(info)
    while status["state"] != "completed" or status["index"] < last_payload_size-1:
        time.sleep(1)
        info = action_status_mo.get_info()
        while info == "" or info is None:
            time.sleep(1)
            info = action_status_mo.get_info()
        status = json.loads(info)

    action = random.choice(actions)
    command = commands[action]

    last_id += 1
    last_payload_size = len(command)
    payload = CommandPayload(last_id, command)

    action_command_mo.set_info(asdict(payload))

    while status["id"] != last_id:
        info = action_status_mo.get_info()
        while info == "" or info is None:
            time.sleep(1)
            info = action_status_mo.get_info()
        status = json.loads(info)
        time.sleep(1)

KeyboardInterrupt: 